In [1]:
import pandas as pd
import numpy as np

In [2]:
# I aggregate PFAS results to the monitoring-location level.
# This table is designed for both summary tables and map popups.
#
# Definitions:
#
# testing_instances
#     Number of PFAS result records at the location.
#Testing instances means number of PFAS result rows at that location. So if one site had 3 sampling dates and 40 PFAS chemicals tested each time, 
#testing instances could be 120.
#
# parameters_tested
#     Number of unique PFAS parameters tested at the location.
#
# parameters_positive
#     Number of unique PFAS parameters detected at least once.
#
# sampling_events
#     Number of unique sampling dates at the location.
#Sampling events means number of unique sampling dates at that location. So if the site was sampled on 3 different days, sampling events = 3
#
# detection_sampling_event_category
#     Whether PFAS was detected in 100% of sampling events or
#     in less than 100% of sampling events.


In [3]:
file_path = "G:/My Drive/Projects/Large projects/PFAS in water/"

data_surface = pd.read_csv(
    file_path + "saved files/Notebook2/pfas_cleaned_analysis_file.csv",
    low_memory=False
)

print(data_surface.shape)
display(data_surface.head())

(95520, 144)


,ActivityBottomDepthHeightMeasure/MeasureUnitCode,ActivityBottomDepthHeightMeasure/MeasureValue,ActivityCommentText,ActivityConductingOrganizationText,ActivityDepthAltitudeReferencePointText,ActivityDepthHeightMeasure/MeasureUnitCode,ActivityDepthHeightMeasure/MeasureValue,ActivityEndDate,ActivityEndTime/Time,ActivityEndTime/TimeZoneCode,...,DEPTH_UNITS,SAMPLE_TYPE_QUALIFIER,LAB_COMMENT,SAMPLE_COMMENT,PARAMETER_SEQ,final_latitude,final_longitude,MaineParameterShortened,MaineMDL_ng_L,ResultMeasureValue_num
0,NaN,NaN,NaN,U.S. Geological Survey-Water Resources Discipline,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,U.S. Geological Survey-Water Resources Discipline,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,U.S. Geological Survey-Water Resources Discipline,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,U.S. Geological Survey-Water Resources Discipline,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,U.S. Geological Survey-Water Resources Discipline,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Standardizing analytes

In [4]:
#New
# I standardize PFAS analyte names before counting unique parameters.
# This prevents the same analyte from being counted separately because of
# capitalization, spacing, or minor naming differences.

data_surface["pfas_analyte_standardized"] = (
    data_surface["CharacteristicName"]
    .astype("string")
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

display(
    data_surface[
        [
            "CharacteristicName",
            "pfas_analyte_standardized"
        ]
    ]
    .drop_duplicates()
    .sort_values("pfas_analyte_standardized")
)

,CharacteristicName,pfas_analyte_standardized
11936,"1-Butanesulfonic acid, 1,1,2,2,3,3,4,4,4-nonaf...","1-BUTANESULFONIC ACID, 1,1,2,2,3,3,4,4,4-NONAF..."
11749,"1-Decanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6...","1-DECANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6..."
11634,"1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,...","1-HEPTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,..."
12049,"1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,...","1-HEPTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,..."
11808,"1-Hexanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6...","1-HEXANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6..."
...,...,...
11660,Sodium Perfluorononanesulfonate,SODIUM PERFLUORONONANESULFONATE
11637,Sodium perfluoropentanesulfonate,SODIUM PERFLUOROPENTANESULFONATE
11643,Sulfluramid,SULFLURAMID
16458,"Tridecanoic acid, 2,2,3,3,4,4,5,5,6,6,7,7,8,8,...","TRIDECANOIC ACID, 2,2,3,3,4,4,5,5,6,6,7,7,8,8,..."


In [5]:
# I replace retired WQP analyte names with the current analyte name
# provided in the CharacteristicName field.

retired_mask = (
    data_surface["pfas_analyte_standardized"]
    .str.contains(
        "RETIRED",
        case=False,
        na=False
    )
)

print(
    "Rows with retired analyte names:",
    retired_mask.sum()
)

display(
    data_surface.loc[
        retired_mask,
        [
            "CharacteristicName",
            "pfas_analyte_standardized"
        ]
    ]
    .drop_duplicates()
)

Rows with retired analyte names: 206


,CharacteristicName,pfas_analyte_standardized
11649,"Undecanoic acid, 2,2,3,3,4,4,5,5,6,6,7,7,8,8,9...","UNDECANOIC ACID, 2,2,3,3,4,4,5,5,6,6,7,7,8,8,9..."
11711,"Decanoic acid, 2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9...","DECANOIC ACID, 2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9..."
11794,Heptafluorobutyric acid***retired***use Perflu...,HEPTAFLUOROBUTYRIC ACID***RETIRED***USE PERFLU...
12394,"1-Hexanesulfonic acid, 3,3,4,4,5,5,6,6,6-nonaf...","1-HEXANESULFONIC ACID, 3,3,4,4,5,5,6,6,6-NONAF..."
16458,"Tridecanoic acid, 2,2,3,3,4,4,5,5,6,6,7,7,8,8,...","TRIDECANOIC ACID, 2,2,3,3,4,4,5,5,6,6,7,7,8,8,..."
32699,Perfluoro-1-octanesulfonamide***retired***use ...,PERFLUORO-1-OCTANESULFONAMIDE***RETIRED***USE ...
32946,"Hexanoic acid, 4,4,5,5,6,6,6-heptafluoro-***re...","HEXANOIC ACID, 4,4,5,5,6,6,6-HEPTAFLUORO-***RE..."
38659,"Nonafluoro-3,6-dioxaheptanoic acid***retired**...","NONAFLUORO-3,6-DIOXAHEPTANOIC ACID***RETIRED**..."
39268,"1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8...","1-OCTANESULFONIC ACID, 3,3,4,4,5,5,6,6,7,7,8,8..."


In [6]:
# I replace retired analyte names with the current name listed
# after "use" in the WQP CharacteristicName.

data_surface.loc[
    retired_mask,
    "pfas_analyte_standardized"
] = (
    data_surface.loc[
        retired_mask,
        "pfas_analyte_standardized"
    ]
    .str.extract(
        r"(?i)USE\s+(.+)$",
        expand=False
    )
    .str.upper()
    .str.strip()
)

In [7]:
# I confirm that no retired analyte names remain.

remaining_retired = (
    data_surface["pfas_analyte_standardized"]
    .str.contains(
        "RETIRED",
        case=False,
        na=False
    )
)

print(
    "Retired analyte names remaining:",
    remaining_retired.sum()
)

Retired analyte names remaining: 0


In [8]:
# I review the unique standardized PFAS analyte names by source
# after fixing retired WQP names.

for source_name in data_surface["source"].dropna().unique():
    
    analytes = (
        data_surface.loc[
            data_surface["source"].eq(source_name),
            "pfas_analyte_standardized"
        ]
        .dropna()
        .drop_duplicates()
        .sort_values()
    )
    
    print("\nSOURCE:", source_name)
    print("Unique standardized analytes:", len(analytes))
    
    display(
        analytes.reset_index(drop=True)
    )


SOURCE: WQP
Unique standardized analytes: 112


0      1-BUTANESULFONIC ACID, 1,1,2,2,3,3,4,4,4-NONAF...
1      1-DECANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6...
2      1-HEPTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,...
3      1-HEPTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,...
4      1-HEXANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6...
                             ...                        
107                      SODIUM PERFLUORODECANESULFONATE
108                      SODIUM PERFLUOROHEXANESULFONATE
109                      SODIUM PERFLUORONONANESULFONATE
110                     SODIUM PERFLUOROPENTANESULFONATE
111                                          SULFLURAMID
Name: pfas_analyte_standardized, Length: 112, dtype: string


SOURCE: Massachusetts
Unique standardized analytes: 67


0     11CL-PF3OUDS
1         3:3 FTCA
2          4:2 FTS
3         5:3 FTCA
4          6:2 FTS
          ...     
62           PFPEA
63           PFPES
64          PFTEDA
65          PFTRDA
66           PFUNA
Name: pfas_analyte_standardized, Length: 67, dtype: string


SOURCE: Maine DEP
Unique standardized analytes: 45


0     11-CHLOROEICOSAFLUORO-3-OXAUNDECANE-1-SULFONIC...
1                     3:3 FLUOROTELOMER CARBOXYLIC ACID
2                   4,8-DIOXA-3H-PERFLUORONONANOIC ACID
3                           4:2 FLUOROTELOMER SULFONATE
4                       4:2-FLUOROTELOMER SULFONIC ACID
5                     5:3 FLUOROTELOMER CARBOXYLIC ACID
6                           6:2 FLUOROTELOMER SULFONATE
7                       6:2 FLUOROTELOMER SULFONIC ACID
8                     7:3 FLUOROTELOMER CARBOXYLIC ACID
9                           8:2 FLUOROTELOMER SULFONATE
10                      8:2 FLUOROTELOMER SULFONIC ACID
11    9-CHOLOROHEXADECAFLUORO-3-OXANONANE-1-SULFONIC...
12                 HEXAFLUOROPROPYLENE OXIDE DIMER ACID
13                  N-ETHYL PERFLUOROOCTANE SULFONAMIDE
14       N-ETHYL PERFLUOROOCTANE SULFONAMIDOACETIC ACID
15           N-ETHYL PERFLUOROOCTANE SULFONAMIDOETHANOL
16                 N-METHYL PERFLUOROOCTANE SULFONAMIDE
17      N-METHYL PERFLUOROOCTANE SULFONAMIDOACET


SOURCE: Waterkeeper Phase I
Unique standardized analytes: 54


0           10:2 FTS
1       11CL-PF3OUDS
2           3:3 FTCA
3            4:2 FTS
4           5:3 FTCA
5            6:2 FTS
6            6:6PFPI
7            6:8PFPI
8           7:3 FTCA
9            8:2 FTS
10          8CL-PFOS
11        9CL-PF3ONS
12             ADONA
13              FBSA
14            FDSA-I
15             FDUEA
16             FHXSA
17             FOSAA
18             FOUEA
19    HFPO-DA (GENX)
20            MEFBSA
21        N-AP-FHXSA
22          N-ETFOSA
23         N-ETFOSAA
24          N-MEFOSA
25         N-MEFOSAA
26           NETFOSE
27             NFDHA
28           NMEFOSE
29              PFBA
30              PFBS
31              PFDA
32             PFDOA
33             PFDOS
34              PFDS
35            PFECHS
36            PFEESA
37             PFHPA
38             PFHPS
39             PFHXA
40             PFHXS
41            PFMOBA
42           PFMOPRA
43              PFNA
44              PFNS
45              PFOA
46              PFOS
47           


SOURCE: Waterkeeper Phase II
Unique standardized analytes: 40


0                                    11CL-PF3OUDS
1                                        3:3 FTCA
2     4,8-DIOXA-3H-PERFLUORONONANOIC ACID (ADONA)
3                                         4:2 FTS
4                                        5:3 FTCA
5                                         6:2 FTS
6                                        7:3 FTCA
7                                         8:2 FTS
8                                      9CL-PF3ONS
9                                  HFPO-DA (GENX)
10                                        NETFOSA
11                                       NETFOSAA
12                                        NETFOSE
13                                          NFDHA
14                                        NMEFOSA
15                                       NMEFOSAA
16                                        NMEFOSE
17            PERFLUOROBUTANESULFONIC ACID (PFBS)
18                  PERFLUOROBUTANOIC ACID (PFBA)
19            PERFLUORODECANESULFONIC ACID (PFDS)


In [9]:
# Waterkeeper Phase II full names -> standardized abbreviations

phase2_analyte_crosswalk = {
    "PERFLUOROBUTANESULFONIC ACID (PFBS)": "PFBS",
    "PERFLUOROBUTANOIC ACID (PFBA)": "PFBA",
    "PERFLUORODECANOIC ACID (PFDA)": "PFDA",
    "PERFLUORODODECANOIC ACID (PFDOA)": "PFDOA",
    "PERFLUOROHEPTANESULFONIC ACID (PFHPS)": "PFHPS",
    "PERFLUOROHEPTANOIC ACID (PFHPA)": "PFHPA",
    "PERFLUOROHEXANESULFONIC ACID (PFHXS)": "PFHXS",
    "PERFLUOROHEXANOIC ACID (PFHXA)": "PFHXA",
    "PERFLUORONONANOIC ACID (PFNA)": "PFNA",
    "PERFLUOROOCTANESULFONAMIDE (FOSA)": "PFOSA",
    "PERFLUOROOCTANESULFONIC ACID (PFOS)": "PFOS",
    "PERFLUOROOCTANOIC ACID (PFOA)": "PFOA",
    "PERFLUOROPENTANESULFONIC ACID (PFPES)": "PFPES",
    "PERFLUOROPENTANOIC ACID (PFPEA)": "PFPEA",
    "PERFLUOROTETRADECANOIC ACID (PFTEA)": "PFTEA",
    "PERFLUOROUNDECANOIC ACID (PFUNA)": "PFUNA"
}

data_surface["pfas_analyte_standardized"] = (
    data_surface["pfas_analyte_standardized"]
    .replace(phase2_analyte_crosswalk)
)

In [10]:
# I display all unique WQP PFAS analyte names so I can
# review potential synonyms and non-PFAS records.

wqp_analytes = (
    data_surface.loc[
        data_surface["source"].eq("WQP"),
        "pfas_analyte_standardized"
    ]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(wqp_analytes)

print(
    "Total unique WQP analyte names:",
    len(wqp_analytes)
)

0                                                                                                         1-BUTANESULFONIC ACID, 1,1,2,2,3,3,4,4,4-NONAFLUORO-, POTASSIUM SALT (1:1)
1                                                                                               1-DECANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,10-HENEICOSAFLUORO-
2                                                                                                             1-HEPTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-PENTADECAFLUORO-
3                                                                                           1-HEPTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-PENTADECAFLUORO-,SODIUM SALT (1:1)
4                                                                                                           1-HEXANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,6-TRIDECAFLUORO-, ION(1-)
5      1-OCTANESULFONAMIDE, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-HEPTADECAFLUORO-N-METHYL-, REACTI

Total unique WQP analyte names: 112


In [11]:
# I standardize clear WQP PFAS synonyms so that acid, anion,
# and salt names for the same PFAS are counted as one analyte.

wqp_analyte_crosswalk = {

    # PFBA
    "PERFLUOROBUTANOATE": "PFBA",
    "PERFLUOROBUTANOIC ACID": "PFBA",

    # PFBS
    "PERFLUOROBUTANESULFONATE": "PFBS",
    "PERFLUOROBUTANESULFONIC ACID": "PFBS",
    "1-BUTANESULFONIC ACID, 1,1,2,2,3,3,4,4,4-NONAFLUORO-, POTASSIUM SALT (1:1)": "PFBS-K",

    # PFPeA
    "PERFLUOROPENTANOATE": "PFPEA",
    "PERFLUOROPENTANOIC ACID": "PFPEA",
    "PERFLUOROVALERIC ACID": "PFPEA",

    # PFPeS
    "PERFLUOROPENTANESULFONATE": "PFPES",
    "PERFLUOROPENTANESULFONIC ACID": "PFPES",
    "SODIUM PERFLUOROPENTANESULFONATE": "PFPES",

    # PFHxA
    "PERFLUOROHEXANOATE": "PFHXA",
    "PERFLUOROHEXANOIC ACID": "PFHXA",

    # PFHxS
    "PERFLUOROHEXANESULFONATE": "PFHXS",
    "PERFLUOROHEXANESULFONIC ACID": "PFHXS",
    "SODIUM PERFLUOROHEXANESULFONATE": "PFHXS",
    "1-HEXANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,6-TRIDECAFLUORO-, ION(1-)": "PFHXS",

    # PFHpA
    "PERFLUOROHEPTANOATE": "PFHPA",
    "PERFLUOROHEPTANOIC ACID": "PFHPA",

    # PFHpS
    "PERFLUOROHEPTANESULFONATE": "PFHPS",

    # PFOA
    "PERFLUOROOCTANOIC ACID": "PFOA",
    "PFOA ION": "PFOA",

    # PFOS
    "PERFLUOROOCTANESULFONATE": "PFOS",
    "PERFLUOROOCTANE SULFONIC ACID": "PFOS",

    # PFOSA / FOSA
    "PERFLUOROOCTANESULFONAMIDE": "PFOSA",

    # PFNA
    "PERFLUORONONANOATE": "PFNA",
    "PERFLUORONONANOIC ACID": "PFNA",

    # PFNS
    "PERFLUORONONANESULFONATE": "PFNS",
    "PERFLUORO-1-NONANESULFONIC ACID": "PFNS",
    "SODIUM PERFLUORONONANESULFONATE": "PFNS",

    # PFDA
    "PERFLUORODECANOATE": "PFDA",
    "PERFLUORODECANOIC ACID": "PFDA",

    # PFDS
    "PERFLUORODECANESULFONATE": "PFDS",
    "SODIUM PERFLUORODECANESULFONATE": "PFDS",
    "1-DECANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,10-HENEICOSAFLUORO-": "PFDS",

    # PFUnA
    "PERFLUOROUNDECANOATE": "PFUNA",
    "PERFLUOROUNDECANOIC ACID": "PFUNA",

    # PFDoA
    "PERFLUORODODECANOATE": "PFDOA",
    "PERFLUORODODECANOIC ACID": "PFDOA",

    # PFDoS
    "PERFLUORODODECANESULFONATE": "PFDOS",
    "PERFLUORODODECANESULFONIC ACID": "PFDOS",

    # PFTrDA
    "PERFLUOROTRIDECANOATE": "PFTRDA",
    "PERFLUOROTRIDECANOIC ACID": "PFTRDA",

    # PFTeA
    "PERFLUOROTETRADECANOATE": "PFTEA",
    "PERFLUOROTETRADECANOIC ACID": "PFTEA",

    # 4:2 FTS
    "FLUOROTELOMER SULFONATE 4:2": "4:2 FTS",
    "FTS 4:2 ION": "4:2 FTS",

    # 6:2 FTS
    "6:2 FLUOROTELOMER SULFONATE ACID": "6:2 FTS",
    "FTS 6:2 ION": "6:2 FTS",

    # 8:2 FTS
    "FLUOROTELOMER SULFONATE 8:2": "8:2 FTS",
    "FTS 8:2 ION": "8:2 FTS",

    # 3:3 FTCA
    "3:3 FLUOROTELOMER CARBOXYLIC ACID": "3:3 FTCA",
    "3:3 FLUOROTELOMER CARBOXYLATE, ION(1-)": "3:3 FTCA",

    # HFPO-DA / GenX
    "HEXAFLUOROPROPYLENE OXIDE DIMER ACID": "HFPO-DA (GENX)",
    "HEXAFLUOROPROPYLENE OXIDE-DIMER ACID": "HFPO-DA (GENX)"
}

# I apply these mappings only to WQP records.

mask_wqp = data_surface["source"].eq("WQP")

data_surface.loc[
    mask_wqp,
    "pfas_analyte_standardized"
] = (
    data_surface.loc[
        mask_wqp,
        "pfas_analyte_standardized"
    ]
    .replace(wqp_analyte_crosswalk)
)

In [12]:
# I check how many unique WQP analytes remain after standardization.

wqp_analytes_after = (
    data_surface.loc[
        data_surface["source"].eq("WQP"),
        "pfas_analyte_standardized"
    ]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print(
    "Unique WQP analytes after standardization:",
    len(wqp_analytes_after)
)

display(
    wqp_analytes_after.reset_index(drop=True)
)

Unique WQP analytes after standardization: 83


0                                                                                                            1-HEPTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-PENTADECAFLUORO-
1                                                                                          1-HEPTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-PENTADECAFLUORO-,SODIUM SALT (1:1)
2     1-OCTANESULFONAMIDE, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-HEPTADECAFLUORO-N-METHYL-, REACTION PRODUCTS WITH BENZENE-CHLORINE-SULFUR CHLORIDE (S2CL2) REACTION PRODUCTS CHLORIDES
3                                                                                      1-OCTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-HEPTADECAFLUORO-, SODIUM SALT (1:1)
4                                                                                                                   1-OCTANESULFONIC ACID, 3,3,4,4,5,5,6,6,7,7,8,8,8-TRIDECAFLUORO-
5                                                                                                   

In [13]:
# I define remaining WQP analyte names that may represent
# different reporting forms of the same PFAS compound.

possible_synonym_groups = {

    "11CL-PF3OUDS": [
        "11-CHLOROEICOSAFLUORO-3-OXAUNDECANE-1-SULFONATE",
        "11-CHLOROEICOSAFLUORO-3-OXAUNDECANE-1-SULFONIC ACID"
    ],

    "9CL-PF3ONS": [
        "9-CHLOROHEXADECAFLUORO-3-OXANONANE-1-SULFONATE",
        "9-CHLOROHEXADECAFLUORO-3-OXANONANE-1-SULFONIC ACID",
        "9-CHLOROHEXADECAFLUORO-3-OXANONE-1-SULFONIC ACID"
    ],

    "ADONA": [
        "4,8-DIOXA-3H-PERFLUORONONANOATE",
        "4,8-DIOXA-3H-PERFLUORONONANOIC ACID",
        "AMMONIUM 4,8-DIOXA-3H-PERFLUORONONANOATE"
    ]
}


for standardized_name, names in possible_synonym_groups.items():

    temp = data_surface.loc[
        data_surface["source"].eq("WQP")
        & data_surface["pfas_analyte_standardized"].isin(names),
        [
            "MonitoringLocationIdentifier",
            "pfas_analyte_standardized"
        ]
    ].drop_duplicates()

    overlap = (
        temp
        .groupby("MonitoringLocationIdentifier")
        ["pfas_analyte_standardized"]
        .nunique()
    )

    print(
        standardized_name,
        "- locations containing more than one form:",
        (overlap > 1).sum()
    )

11CL-PF3OUDS - locations containing more than one form: 0
9CL-PF3ONS - locations containing more than one form: 8
ADONA - locations containing more than one form: 0


In [14]:
# I standardize the WQP names for 9Cl-PF3ONS.
#
# Multiple names for this analyte occur at the same monitoring
# locations and would otherwise cause the analyte to be counted
# more than once in parameters_tested.

wqp_9cl_crosswalk = {
    "9-CHLOROHEXADECAFLUORO-3-OXANONANE-1-SULFONATE":
        "9CL-PF3ONS",

    "9-CHLOROHEXADECAFLUORO-3-OXANONANE-1-SULFONIC ACID":
        "9CL-PF3ONS",

    "9-CHLOROHEXADECAFLUORO-3-OXANONE-1-SULFONIC ACID":
        "9CL-PF3ONS"
}

mask_wqp = data_surface["source"].eq("WQP")

data_surface.loc[
    mask_wqp,
    "pfas_analyte_standardized"
] = (
    data_surface.loc[
        mask_wqp,
        "pfas_analyte_standardized"
    ]
    .replace(wqp_9cl_crosswalk)
)

In [15]:
# I confirm that the different WQP names for 9Cl-PF3ONS
# have been combined into one standardized analyte.

display(
    data_surface.loc[
        data_surface["source"].eq("WQP")
        & data_surface["pfas_analyte_standardized"].eq("9CL-PF3ONS"),
        "CharacteristicName"
    ]
    .value_counts()
    .reset_index(name="records")
)

print(
    "Unique standardized analytes:",
    data_surface.loc[
        data_surface["source"].eq("WQP")
        & data_surface["pfas_analyte_standardized"].eq("9CL-PF3ONS"),
        "pfas_analyte_standardized"
    ].nunique()
)

,CharacteristicName,records
0,9-Chlorohexadecafluoro-3-oxanonane-1-sulfonate,737
1,9-Chlorohexadecafluoro-3-oxanone-1-sulfonic acid,688
2,9-Chlorohexadecafluoro-3-oxanonane-1-sulfonic acid,164


Unique standardized analytes: 1


In [16]:
# I compare analyte names across data sources.

analyte_by_source = (
    data_surface[
        [
            "source",
            "CharacteristicName"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "CharacteristicName",
            "source"
        ]
    )
)

display(analyte_by_source)

,source,CharacteristicName
11936,WQP,"1-Butanesulfonic acid, 1,1,2,2,3,3,4,4,4-nonafluoro-, potassium salt (1:1)"
11749,WQP,"1-Decanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,10-heneicosafluoro-"
11634,WQP,"1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-pentadecafluoro-"
12049,WQP,"1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-pentadecafluoro-,sodium salt (1:1)"
11808,WQP,"1-Hexanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,6-tridecafluoro-, ion(1-)"
12394,WQP,"1-Hexanesulfonic acid, 3,3,4,4,5,5,6,6,6-nonafluoro-***retired***use Fluorotelomer sulfonate 4:2"
35237,WQP,"1-Octanesulfonamide, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluoro-N-methyl-, reaction products with benzene-chlorine-sulfur chloride (S2Cl2) reaction products chlorides"
11918,WQP,"1-Octanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluoro-, sodium salt (1:1)"
30,WQP,"1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8,8-tridecafluoro-"
39268,WQP,"1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8,8-tridecafluoro-***retired***use 6:2 Fluorotelomer sulfonate acid"


In [17]:
# I count how many sources report each analyte name.

analyte_source_count = (
    data_surface[
        [
            "source",
            "CharacteristicName"
        ]
    ]
    .drop_duplicates()
    .groupby("CharacteristicName")
    .size()
    .reset_index(name="number_of_sources")
    .sort_values(
        [
            "number_of_sources",
            "CharacteristicName"
        ]
    )
)

display(analyte_source_count)

,CharacteristicName,number_of_sources
0,"1-Butanesulfonic acid, 1,1,2,2,3,3,4,4,4-nonafluoro-, potassium salt (1:1)",1
1,"1-Decanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,10-heneicosafluoro-",1
2,"1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-pentadecafluoro-",1
3,"1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-pentadecafluoro-,sodium salt (1:1)",1
4,"1-Hexanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,6-tridecafluoro-, ion(1-)",1
5,"1-Hexanesulfonic acid, 3,3,4,4,5,5,6,6,6-nonafluoro-***retired***use Fluorotelomer sulfonate 4:2",1
6,"1-Octanesulfonamide, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluoro-N-methyl-, reaction products with benzene-chlorine-sulfur chloride (S2Cl2) reaction products chlorides",1
7,"1-Octanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluoro-, sodium salt (1:1)",1
8,"1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8,8-tridecafluoro-",1
9,"1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8,8-tridecafluoro-***retired***use 6:2 Fluorotelomer sulfonate acid",1


In [18]:
# I compare analyte names by source.

analyte_source_table = (
    data_surface[
        [
            "CharacteristicName",
            "source"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "CharacteristicName",
            "source"
        ]
    )
)

display(analyte_source_table)

,CharacteristicName,source
11936,"1-Butanesulfonic acid, 1,1,2,2,3,3,4,4,4-nonafluoro-, potassium salt (1:1)",WQP
11749,"1-Decanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,10-heneicosafluoro-",WQP
11634,"1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-pentadecafluoro-",WQP
12049,"1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-pentadecafluoro-,sodium salt (1:1)",WQP
11808,"1-Hexanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,6-tridecafluoro-, ion(1-)",WQP
12394,"1-Hexanesulfonic acid, 3,3,4,4,5,5,6,6,6-nonafluoro-***retired***use Fluorotelomer sulfonate 4:2",WQP
35237,"1-Octanesulfonamide, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluoro-N-methyl-, reaction products with benzene-chlorine-sulfur chloride (S2Cl2) reaction products chlorides",WQP
11918,"1-Octanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluoro-, sodium salt (1:1)",WQP
30,"1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8,8-tridecafluoro-",WQP
39268,"1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8,8-tridecafluoro-***retired***use 6:2 Fluorotelomer sulfonate acid",WQP


In [19]:
print(
    sorted(
        data_surface.loc[
            data_surface["source"] == "Waterkeeper Phase II",
            "CharacteristicName"
        ].dropna().unique()
    )
)

['11Cl-PF3OUdS', '3:3 FTCA', '4,8-Dioxa-3H-perfluorononanoic acid (ADONA)', '4:2 FTS', '5:3 FTCA', '6:2 FTS', '7:3 FTCA', '8:2 FTS', '9Cl-PF3ONS', 'HFPO-DA (GenX)', 'NEtFOSA', 'NEtFOSAA', 'NEtFOSE', 'NFDHA', 'NMeFOSA', 'NMeFOSAA', 'NMeFOSE', 'PFEESA', 'PFMBA', 'PFMPA', 'Perfluorobutanesulfonic acid (PFBS)', 'Perfluorobutanoic acid (PFBA)', 'Perfluorodecanesulfonic acid (PFDS)', 'Perfluorodecanoic acid (PFDA)', 'Perfluorododecanesulfonic acid (PFDoS)', 'Perfluorododecanoic acid (PFDoA)', 'Perfluoroheptanesulfonic acid (PFHpS)', 'Perfluoroheptanoic acid (PFHpA)', 'Perfluorohexanesulfonic acid (PFHxS)', 'Perfluorohexanoic acid (PFHxA)', 'Perfluorononanesulfonic acid (PFNS)', 'Perfluorononanoic acid (PFNA)', 'Perfluorooctanesulfonamide (FOSA)', 'Perfluorooctanesulfonic acid (PFOS)', 'Perfluorooctanoic acid (PFOA)', 'Perfluoropentanesulfonic acid (PFPeS)', 'Perfluoropentanoic acid (PFPeA)', 'Perfluorotetradecanoic acid (PFTeA)', 'Perfluorotridecanoic acid (PFTrDA)', 'Perfluoroundecanoic aci

In [20]:
print(
    sorted(
        data_surface.loc[
            data_surface["source"] == "Waterkeeper Phase I",
            "CharacteristicName"
        ].dropna().unique()
    )
)

['10:2 FTS', '11CL-PF3OUdS', '3:3 FTCA', '4:2 FTS', '5:3 FTCA', '6:2 FTS', '6:6PFPi', '6:8PFPi', '7:3 FTCA', '8:2 FTS', '8Cl-PFOS', '9Cl-PF3ONS', 'ADONA', 'FBSA', 'FDSA-I', 'FDUEA', 'FHxSA', 'FOSAA', 'FOUEA', 'HFPO-DA\n(GenX)', 'MeFBSA', 'N-AP-FHxSA', 'N-EtFOSA', 'N-EtFOSAA', 'N-MeFOSA', 'N-MeFOSAA', 'NEtFOSE', 'NFDHA', 'NMeFOSE', 'PFBA', 'PFBS', 'PFDA', 'PFDS', 'PFDoA', 'PFDoS', 'PFECHS', 'PFEESA', 'PFHpA', 'PFHpS', 'PFHxA', 'PFHxS', 'PFMOBA', 'PFMOPrA', 'PFNA', 'PFNS', 'PFOA', 'PFOS', 'PFOSA', 'PFPeA', 'PFPeS', 'PFPrS', 'PFTeA', 'PFTrDA', 'PFUnA']


In [21]:
print(
    sorted(
        data_surface.loc[
            data_surface["source"].str.contains("WQP", na=False),
            "CharacteristicName"
        ].dropna().unique()
    )
)

['1-Butanesulfonic acid, 1,1,2,2,3,3,4,4,4-nonafluoro-, potassium salt (1:1)', '1-Decanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10,10-heneicosafluoro-', '1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-pentadecafluoro-', '1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-pentadecafluoro-,sodium salt (1:1)', '1-Hexanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,6-tridecafluoro-, ion(1-)', '1-Hexanesulfonic acid, 3,3,4,4,5,5,6,6,6-nonafluoro-***retired***use Fluorotelomer sulfonate 4:2', '1-Octanesulfonamide, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluoro-N-methyl-, reaction products with benzene-chlorine-sulfur chloride (S2Cl2) reaction products chlorides', '1-Octanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluoro-, sodium salt (1:1)', '1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8,8-tridecafluoro-', '1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8,8-tridecafluoro-***retired***use 6:2 Fluorotelomer sulfonate acid', '1-Propanesulfonic acid, 1,1,2,2,3,

In [22]:
print(
    sorted(
        data_surface.loc[
            data_surface["source"].str.contains("Massachusetts", na=False),
            "CharacteristicName"
        ].dropna().unique()
    )
)

['11Cl-PF3OUdS', '3:3 FTCA', '4:2 FTS', '5:3 FTCA', '6:2 FTS', '7:3 FTCA', '8:2 FTS', '9Cl-PF3ONS', 'ADONA', 'DONA', 'EtFOSAA', 'HFPO-DA', 'MeFOSAA', 'NEtFOSA', 'NEtFOSAA', 'NEtFOSE', 'NFDHA', 'NMeFOSA', 'NMeFOSAA', 'NMeFOSE', 'PES', 'PFBA', 'PFBS', 'PFDA', 'PFDS', 'PFDoA', 'PFDoS', 'PFECA A', 'PFECA B', 'PFECA F', 'PFEESA', 'PFHpA', 'PFHpS', 'PFHxA', 'PFHxS', 'PFMBA', 'PFMPA', 'PFNA', 'PFNS', 'PFOA', 'PFOS', 'PFOSA', 'PFPeA', 'PFPeS', 'PFTeDA', 'PFTrDA', 'PFUnA', 'Perfluorobutanesulfonic acid', 'Perfluorobutanoic acid', 'Perfluorodecanesulfonic acid', 'Perfluorodecanoic acid', 'Perfluorododecanesulfonic acid (PFDoS)', 'Perfluorododecanoic acid', 'Perfluoroheptanesulfonic acid', 'Perfluoroheptanoic acid', 'Perfluorohexanesulfonic acid', 'Perfluorohexanoic acid', 'Perfluorononanesulfonic acid', 'Perfluorononanoic acid', 'Perfluorooctanesulfonamide', 'Perfluorooctanesulfonic acid', 'Perfluorooctanoic acid', 'Perfluoropentanesulfonic acid', 'Perfluoropentanoic acid', 'Perfluorotetradecano

In [23]:
# I compare Waterkeeper analytes against WQP analytes
# after removing the abbreviations in parentheses.

wqp_names = (
    data_surface.loc[
        data_surface["source"].str.contains("WQP", na=False),
        "CharacteristicName"
    ]
    .dropna()
    .str.upper()
)

waterkeeper_names = (
    data_surface.loc[
        data_surface["source"].isin(
            ["Waterkeeper Phase I", "Waterkeeper Phase II"]
        ),
        "CharacteristicName"
    ]
    .dropna()
    .str.upper()
)

print("WQP analytes:", wqp_names.nunique())
print("Waterkeeper analytes:", waterkeeper_names.nunique())

WQP analytes: 121
Waterkeeper analytes: 82


In [24]:
# I check how many Waterkeeper analytes already appear
# as abbreviations inside WQP analyte names.

wqp_names = (
    data_surface.loc[
        data_surface["source"].str.contains("WQP", na=False),
        "CharacteristicName"
    ]
    .dropna()
    .unique()
)

waterkeeper_names = (
    data_surface.loc[
        data_surface["source"].isin(
            ["Waterkeeper Phase I", "Waterkeeper Phase II"]
        ),
        "CharacteristicName"
    ]
    .dropna()
    .unique()
)

for analyte in sorted(waterkeeper_names):
    
    matches = [
        wqp_name
        for wqp_name in wqp_names
        if f"({analyte})" in str(wqp_name)
    ]
    
    if matches:
        print(f"\n{analyte}")
        for match in matches:
            print("   ", match)

In [25]:
# I apply any remaining manual PFAS analyte mappings without
# overwriting the analyte standardization completed earlier.

analyte_crosswalk = {
    "PFOS": "PFOS",
    "PFOA": "PFOA",
    "PFHXS": "PFHXS",
    "PFHXA": "PFHXA",
    "PFNA": "PFNA",
    "PFDA": "PFDA",
    "PFDOA": "PFDOA",
    "PFHPA": "PFHPA",
    "PFHPS": "PFHPS",
    "PFPEA": "PFPEA",
    "PFPES": "PFPES",
    "PFBA": "PFBA",
    "PFBS": "PFBS",
    "PFUNA": "PFUNA",
    "PFTEA": "PFTEA",
    "PFTRDA": "PFTRDA",
    "PFOSA": "PFOSA",
    "FOSA": "PFOSA",
    "HFPO-DA (GENX)": "HFPO-DA (GENX)",
    "6:2 FTS": "6:2 FTS",
    "8:2 FTS": "8:2 FTS",
    "4:2 FTS": "4:2 FTS",
    "10:2 FTS": "10:2 FTS"
}

data_surface["pfas_analyte_standardized"] = (
    data_surface["pfas_analyte_standardized"]
    .replace(analyte_crosswalk)
)

display(
    data_surface[
        [
            "CharacteristicName",
            "pfas_analyte_standardized",
            "source"
        ]
    ]
    .drop_duplicates()
    .sort_values("pfas_analyte_standardized")
)

,CharacteristicName,pfas_analyte_standardized,source
11634,"1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-pentadecafluoro-","1-HEPTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-PENTADECAFLUORO-",WQP
12049,"1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-pentadecafluoro-,sodium salt (1:1)","1-HEPTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,7-PENTADECAFLUORO-,SODIUM SALT (1:1)",WQP
35237,"1-Octanesulfonamide, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluoro-N-methyl-, reaction products with benzene-chlorine-sulfur chloride (S2Cl2) reaction products chlorides","1-OCTANESULFONAMIDE, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-HEPTADECAFLUORO-N-METHYL-, REACTION PRODUCTS WITH BENZENE-CHLORINE-SULFUR CHLORIDE (S2CL2) REACTION PRODUCTS CHLORIDES",WQP
11918,"1-Octanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluoro-, sodium salt (1:1)","1-OCTANESULFONIC ACID, 1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-HEPTADECAFLUORO-, SODIUM SALT (1:1)",WQP
30,"1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8,8-tridecafluoro-","1-OCTANESULFONIC ACID, 3,3,4,4,5,5,6,6,7,7,8,8,8-TRIDECAFLUORO-",WQP
54270,"1-Propanesulfonic acid, 1,1,2,2,3,3,3-heptafluoro-","1-PROPANESULFONIC ACID, 1,1,2,2,3,3,3-HEPTAFLUORO-",WQP
80329,10:2 FTS,10:2 FTS,Waterkeeper Phase I
20,11-Chloroeicosafluoro-3-oxaundecane-1-sulfonate,11-CHLOROEICOSAFLUORO-3-OXAUNDECANE-1-SULFONATE,WQP
2,11-chloroeicosafluoro-3-oxaundecane-1-sulfonate,11-CHLOROEICOSAFLUORO-3-OXAUNDECANE-1-SULFONATE,WQP
61429,11-CHLOROEICOSAFLUORO-3-OXAUNDECANE-1-SULFONIC ACID,11-CHLOROEICOSAFLUORO-3-OXAUNDECANE-1-SULFONIC ACID,Maine DEP


In [26]:
# I identify standardized analytes that come from multiple original names.

analyte_mapping = (
    data_surface[
        [
            "CharacteristicName",
            "pfas_analyte_standardized"
        ]
    ]
    .drop_duplicates()
)

analyte_mapping_summary = (
    analyte_mapping
    .groupby("pfas_analyte_standardized")
    .size()
    .reset_index(name="original_names")
    .sort_values("original_names", ascending=False)
)

display(
    analyte_mapping_summary.loc[
        analyte_mapping_summary["original_names"] > 1
    ]
)

,pfas_analyte_standardized,original_names
153,PFHXS,6
164,PFPES,5
169,PFUNA,5
163,PFPEA,5
138,PFBA,5
141,PFDA,5
162,PFOSA,4
59,HFPO-DA (GENX),4
24,4:2 FTS,4
18,3:3 FTCA,4


In [27]:
print(
    "Unique CharacteristicName:",
    data_surface["CharacteristicName"].nunique()
)

print(
    "Unique standardized analytes:",
    data_surface["pfas_analyte_standardized"].nunique()
)

Unique CharacteristicName: 263
Unique standardized analytes: 174


In [28]:
print(
    "Unique MonitoringLocationIdentifier:",
    data_surface["MonitoringLocationIdentifier"].nunique()
)

print(
    "Unique MonitoringLocationName:",
    data_surface["MonitoringLocationName"].nunique()
)

Unique MonitoringLocationIdentifier: 1606
Unique MonitoringLocationName: 1459


In [29]:
# Tony said to check monitoring identifier and see if they match the number of name i have 
location_name_check = (
    data_surface
    .groupby("MonitoringLocationName", dropna=False)
    .agg(
        unique_ids=(
            "MonitoringLocationIdentifier",
            "nunique"
        )
    )
    .reset_index()
)

location_name_check["unique_ids"].value_counts().sort_index()

unique_ids
1     1341
2      108
3        4
4        4
10       1
11       1
Name: count, dtype: int64

In [30]:
# I create a unique location table to verify the number of
# unique monitoring locations and coordinates.

location_check = (
    data_surface[
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure"
        ]
    ]
    .drop_duplicates()
)

print("Unique location records:", len(location_check))

display(location_check.head())

Unique location records: 1606


,MonitoringLocationIdentifier,MonitoringLocationName,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure
0,USGS-01465500,"Neshaminy Creek near Langhorne, PA",NaN,NaN
16,USGS-04012500,"POPLAR RIVER AT LUTSEN, MN",NaN,NaN
47,USGS-04044500,"CARP RIVER AT OLD HWY M-28 NEAR MARQUETTE, MI",NaN,NaN
79,USGS-030109575,"Tunungwant Creek below outfall at Bradford, PA",NaN,NaN
116,USGS-01638900,"White Run near Gettysburg, PA",NaN,NaN


In [31]:
duplicate_names = (
    location_name_check
    .loc[location_name_check["unique_ids"] > 1]
    .sort_values("unique_ids", ascending=False)
)

print(
    "Location names linked to multiple identifiers:",
    len(duplicate_names)
)

display(duplicate_names.head(50))

Location names linked to multiple identifiers: 118


,MonitoringLocationName,unique_ids
373,Colorado River,11
1015,"PFAS sampling, ERTC, Sargent Creek",10
528,Field Blank,4
1332,Spokane River,4
1455,Withlacoochee River,4
527,Falls Pond,4
956,Neosho River,3
911,Mill Creek,3
1211,SW-1,3
1215,SW-2,3


In [32]:
# it should be fine

In [33]:
# I assign a placeholder sampling-event value to Waterkeeper Phase II records.

# The original sampling dates were not retained when the Phase II data were
# reshaped. Each Phase II monitoring location represents one sampling event,
# so the placeholder preserves the correct number of sampling events per location.

data_surface["source_clean"] = (
    data_surface["source"]
    .astype("string")
    .str.strip()
)

waterkeeper_missing_date = (
    data_surface["source_clean"].eq("Waterkeeper Phase II")
    & data_surface["ActivityStartDate"].isna()
)

data_surface.loc[
    waterkeeper_missing_date,
    "ActivityStartDate"
] = "Waterkeeper Phase II single sampling event"

print(
    "Waterkeeper Phase II rows assigned placeholder sampling event:",
    waterkeeper_missing_date.sum()
)

print(
    "Remaining Waterkeeper Phase II rows with missing ActivityStartDate:",
    (
        data_surface["source_clean"].eq("Waterkeeper Phase II")
        & data_surface["ActivityStartDate"].isna()
    ).sum()
)

Waterkeeper Phase II rows assigned placeholder sampling event: 2880
Remaining Waterkeeper Phase II rows with missing ActivityStartDate: 0


In [34]:
# I remove Field Blank quality-control samples.
# Tony: this is NEW

field_blank_mask = (
    data_surface["MonitoringLocationName"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("field blank")
)

print("Field Blank records removed:", field_blank_mask.sum())

data_surface = data_surface.loc[
    ~field_blank_mask
].copy()

Field Blank records removed: 280


In [35]:
# I aggregate PFAS results to the monitoring-location level.

#
# Definitions:
#
# testing_instances
# Total number of PFAS testing records at the location.
#
# parameters_tested
# Number of unique PFAS analytes tested at the location.
#
# parameters_positive
# Number of unique PFAS analytes classified as detected ("Y")
# at least once at the location.
#
# sampling_events
# Number of unique sampling dates at the location.
#
# sampling_events_with_detection
# Number of unique sampling dates when at least one PFAS analyte
# was classified as detected ("Y").
#
# max_pfas
# Highest reported numeric PFAS concentration at the location.


site_level = (
    data_surface
    .groupby(
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure",
            "state"
        ],
        dropna=False
    )
    .agg(
        testing_instances=(
            "pfas_detected",
            "size"
        ),

        parameters_tested=(
            "pfas_analyte_standardized",
            "nunique"
        ),

        parameters_positive=(
            "pfas_analyte_standardized",
            lambda x: x[
                data_surface.loc[
                    x.index,
                    "pfas_detected"
                ].eq("Y")
            ].nunique()
        ),

        sampling_events=(
            "ActivityStartDate",
            "nunique"
        ),

        sampling_events_with_detection=(
            "pfas_detected",
            lambda x: (
                data_surface.loc[x.index]
                .groupby("ActivityStartDate")["pfas_detected"]
                .apply(
                    lambda y: y.eq("Y").any()
                )
                .sum()
            )
        ),

        max_pfas=(
            "ResultMeasureValue_num",
            "max"
        )
    )
    .reset_index()
)


# I calculate the percentage of sampling events at each location
# where at least one PFAS analyte was detected.

site_level["percent_sampling_events_with_detection"] = (
    site_level["sampling_events_with_detection"]
    / site_level["sampling_events"]
    * 100
)


# I classify locations based on the share of sampling events
# where at least one PFAS analyte was detected.

site_level["detection_sampling_event_category"] = np.select(
    [
        site_level[
            "percent_sampling_events_with_detection"
        ].eq(100),

        site_level[
            "percent_sampling_events_with_detection"
        ].gt(0)
        & site_level[
            "percent_sampling_events_with_detection"
        ].lt(100),

        site_level[
            "percent_sampling_events_with_detection"
        ].eq(0)
    ],
    [
        "PFAS detected in 100% of sampling events",
        "PFAS detected in less than 100% of sampling events",
        "No PFAS detected in sampling events"
    ],
    default="Unknown"
)

In [36]:
# I identify the PFAS compound associated with the highest
# reported concentration at each monitoring location.

max_pfas_compound = (
    data_surface.loc[
        data_surface["ResultMeasureValue_num"].notna()
    ]
    .sort_values(
        "ResultMeasureValue_num",
        ascending=False
    )
    .drop_duplicates(
        subset="MonitoringLocationIdentifier",
        keep="first"
    )
    [
        [
            "MonitoringLocationIdentifier",
            "pfas_analyte_standardized",
            "ResultMeasureValue_num"
        ]
    ]
    .rename(
        columns={
            "pfas_analyte_standardized": "max_pfas_compound",
            "ResultMeasureValue_num": "max_pfas_check"
        }
    )
)

# I merge the compound associated with the maximum concentration
# into the site-level table.

site_level = site_level.merge(
    max_pfas_compound,
    on="MonitoringLocationIdentifier",
    how="left"
)

# I confirm that the concentration used to identify the compound
# matches the site-level maximum concentration.

print(
    "Rows where max concentration does not match:",
    (
        site_level["max_pfas"].notna()
        & ~site_level["max_pfas"].eq(
            site_level["max_pfas_check"]
        )
    ).sum()
)

display(
    site_level[
        [
            "MonitoringLocationName",
            "state",
            "max_pfas",
            "max_pfas_compound"
        ]
    ]
    .sort_values(
        "max_pfas",
        ascending=False
    )
    .head(20)
)

# The check column is no longer needed.

site_level = site_level.drop(
    columns="max_pfas_check"
)

Rows where max concentration does not match: 0


,MonitoringLocationName,state,max_pfas,max_pfas_compound
632,POND IN,ME,1040000.0,PERFLUOROOCTANE SULFONIC ACID
475,SW-11,ME,197000.0,PERFLUOROOCTANE SULFONIC ACID
476,SW-17,ME,159000.0,PERFLUOROOCTANE SULFONIC ACID
477,SW-23,ME,60200.0,PERFLUOROOCTANE SULFONIC ACID
630,SW-65,ME,43600.0,PERFLUOROOCTANE SULFONIC ACID
655,MEB,ME,39300.0,PERFLUOROOCTANE SULFONIC ACID
113,Lake Holloman Deep,NM,30000.0,PFHXA
806,UPPER DETENTION POND,ME,28600.0,PERFLUOROOCTANE SULFONIC ACID
631,SW-66,ME,14100.0,PERFLUOROOCTANE SULFONIC ACID
653,SW-67,ME,8160.0,PERFLUOROOCTANE SULFONIC ACID


In [37]:
# I review how locations are distributed across the sampling-event categories.

display(
    site_level["detection_sampling_event_category"]
    .value_counts(dropna=False)
    .rename_axis("Detection category")
    .reset_index(name="Number of locations")
)

,Detection category,Number of locations
0,PFAS detected in 100% of sampling events,1280
1,No PFAS detected in sampling events,273
2,PFAS detected in less than 100% of sampling events,49


In [38]:
display(
    data_surface.loc[
        data_surface["MonitoringLocationName"].isin(
            [
                "PFAS sampling, ERTC, Sargent Creek",
                "Field Blank"
            ]
        ),
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure",
            "OrganizationFormalName",
            "state"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "MonitoringLocationName",
            "MonitoringLocationIdentifier"
        ]
    )
)

,MonitoringLocationIdentifier,MonitoringLocationName,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure,OrganizationFormalName,state
11690,MNPCA-PD00027,"PFAS sampling, ERTC, Sargent Creek",46.657940,-92.238730,Minnesota Pollution Control Agency - Ambient Surface Water,MN
11786,MNPCA-PD00028,"PFAS sampling, ERTC, Sargent Creek",46.658433,-92.240102,Minnesota Pollution Control Agency - Ambient Surface Water,MN
11688,MNPCA-PD00029,"PFAS sampling, ERTC, Sargent Creek",46.658481,-92.240161,Minnesota Pollution Control Agency - Ambient Surface Water,MN
11833,MNPCA-PD00030,"PFAS sampling, ERTC, Sargent Creek",46.657380,-92.240030,Minnesota Pollution Control Agency - Ambient Surface Water,MN
11817,MNPCA-PD00031,"PFAS sampling, ERTC, Sargent Creek",46.656595,-92.238608,Minnesota Pollution Control Agency - Ambient Surface Water,MN
11675,MNPCA-S016-782,"PFAS sampling, ERTC, Sargent Creek",46.657839,-92.238361,Minnesota Pollution Control Agency - Ambient Surface Water,MN
11646,MNPCA-S016-787,"PFAS sampling, ERTC, Sargent Creek",46.653656,-92.227015,Minnesota Pollution Control Agency - Ambient Surface Water,MN
11795,MNPCA-S016-788,"PFAS sampling, ERTC, Sargent Creek",46.657026,-92.235846,Minnesota Pollution Control Agency - Ambient Surface Water,MN
11810,MNPCA-S016-789,"PFAS sampling, ERTC, Sargent Creek",46.657068,-92.235325,Minnesota Pollution Control Agency - Ambient Surface Water,MN
11662,MNPCA-S016-790,"PFAS sampling, ERTC, Sargent Creek",46.656828,-92.232522,Minnesota Pollution Control Agency - Ambient Surface Water,MN


In [39]:
# Tony: I reviewed the duplicated MonitoringLocationNames. 
#The difference is expected because many agencies use the same descriptive name for multiple monitoring stations. 
#Examples include large rivers such as the Colorado River and Charles River, as well as study areas such as Sargent Creek where multiple monitoring points share the same location name. 

In [40]:
# I create a location-level PFAS detection flag.
#
# Y means at least one PFAS result was detected at the location.
# U means no PFAS result was detected, but at least one result was uncertain.
# N means all PFAS results at the location were classified as not detected.

site_detection_status = (
    data_surface
    .groupby(
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure",
            "state"
        ],
        dropna=False
    )
    .agg(
        detected_results=(
            "pfas_detected",
            lambda x: x.eq("Y").sum()
        ),
        uncertain_results=(
            "pfas_detected",
            lambda x: x.eq("U").sum()
        ),
        not_detected_results=(
            "pfas_detected",
            lambda x: x.eq("N").sum()
        )
    )
    .reset_index()
)

site_level = site_level.merge(
    site_detection_status,
    on=[
        "MonitoringLocationIdentifier",
        "MonitoringLocationName",
        "ActivityLocation/LatitudeMeasure",
        "ActivityLocation/LongitudeMeasure",
        "state"
    ],
    how="left"
)

site_level["location_pfas_detected"] = np.select(
    [
        site_level["detected_results"] > 0,
        site_level["uncertain_results"] > 0,
        site_level["not_detected_results"] > 0
    ],
    [
        "Y",
        "U",
        "N"
    ],
    default="Unknown"
)

display(
    site_level["location_pfas_detected"]
    .value_counts(dropna=False)
)

location_pfas_detected
Y    1329
N     273
Name: count, dtype: int64

In [41]:
# I create a summary table based on location-level detection status.

location_detection_summary = (
    site_level
    .groupby(
        "location_pfas_detected",
        dropna=False
    )
    .agg(
        locations=(
            "MonitoringLocationIdentifier",
            "nunique"
        ),
        testing_instances=(
            "testing_instances",
            "sum"
        ),
        sampling_events=(
            "sampling_events",
            "sum"
        ),
        sampling_events_with_detection=(
            "sampling_events_with_detection",
            "sum"
        ),
        max_pfas=(
            "max_pfas",
            "max"
        )
    )
    .reset_index()
)


# I calculate the percentage of monitoring locations
# in each detection category.

location_detection_summary["percent_locations"] = (
    location_detection_summary["locations"]
    / site_level["MonitoringLocationIdentifier"].nunique()
    * 100
).round(1)


# I add the location-level detection status back to the
# record-level dataset so I can calculate unique PFAS analytes
# for each location detection category.

data_with_location_status = data_surface.merge(
    site_level[
        [
            "MonitoringLocationIdentifier",
            "location_pfas_detected"
        ]
    ].drop_duplicates(),
    on="MonitoringLocationIdentifier",
    how="left"
)


# I count each PFAS analyte only once within each
# location detection category.

parameter_summary = (
    data_with_location_status
    .groupby(
        "location_pfas_detected",
        dropna=False
    )
    .agg(
        parameters_tested=(
            "pfas_analyte_standardized",
            "nunique"
        )
    )
    .reset_index()
)


# I count each detected PFAS analyte only once within each
# location detection category.

positive_parameter_summary = (
    data_with_location_status.loc[
        data_with_location_status["pfas_detected"].eq("Y")
    ]
    .groupby(
        "location_pfas_detected",
        dropna=False
    )
    ["pfas_analyte_standardized"]
    .nunique()
    .reset_index(
        name="parameters_positive"
    )
)


# I combine the unique PFAS parameter counts with
# the location summary.

location_detection_summary = (
    location_detection_summary
    .merge(
        parameter_summary,
        on="location_pfas_detected",
        how="left"
    )
    .merge(
        positive_parameter_summary,
        on="location_pfas_detected",
        how="left"
    )
)

location_detection_summary["parameters_positive"] = (
    location_detection_summary["parameters_positive"]
    .fillna(0)
    .astype(int)
)

display(location_detection_summary)

,location_pfas_detected,locations,testing_instances,sampling_events,sampling_events_with_detection,max_pfas,percent_locations,parameters_tested,parameters_positive
0,N,273,8327,363,0,NaN,17.0,140,0
1,Y,1329,86913,2324,2253,1040000.0,83.0,174,114


In [42]:
location_detection_summary.to_csv(
    file_path+"saved files/Notebook3/national_level.csv",
    index=False
)
print("Saved updated national_level.csv")

Saved updated national_level.csv


In [43]:
site_level.to_csv(
    file_path + "saved files/Notebook3/pfas_site_level.csv",
    index=False
)

print("Saved updated pfas_site_level.csv")
print("Rows:", len(site_level))

Saved updated pfas_site_level.csv
Rows: 1602


In [44]:
# I create a summary table based on sampling-level PFAS detection.
#
# This table separates locations into:
# - PFAS detected in 100% of sampling
# - PFAS detected in less than 100% of sampling
# - No PFAS detected

sampling_category_summary = (
    site_level
    .groupby(
        "detection_sampling_event_category",
        dropna=False
    )
    .agg(
        locations=(
            "MonitoringLocationIdentifier",
            "nunique"
        ),
        testing_instances=(
            "testing_instances",
            "sum"
        ),
        sampling_events=(
            "sampling_events",
            "sum"
        ),
        sampling_events_with_detection=(
            "sampling_events_with_detection",
            "sum"
        ),
        max_pfas=(
            "max_pfas",
            "max"
        )
    )
    .reset_index()
)


# I calculate the percentage of monitoring locations
# in each sampling detection category.

sampling_category_summary["percent_locations"] = (
    sampling_category_summary["locations"]
    / site_level["MonitoringLocationIdentifier"].nunique()
    * 100
).round(1)


# I add each location's sampling detection category back to
# the record-level dataset.

data_with_sampling_category = data_surface.merge(
    site_level[
        [
            "MonitoringLocationIdentifier",
            "detection_sampling_event_category"
        ]
    ].drop_duplicates(),
    on="MonitoringLocationIdentifier",
    how="left"
)


# I count each PFAS analyte only once within each
# sampling detection category.

sampling_parameter_summary = (
    data_with_sampling_category
    .groupby(
        "detection_sampling_event_category",
        dropna=False
    )
    .agg(
        parameters_tested=(
            "pfas_analyte_standardized",
            "nunique"
        )
    )
    .reset_index()
)


# I count each detected PFAS analyte only once within each
# sampling detection category.

sampling_positive_parameter_summary = (
    data_with_sampling_category.loc[
        data_with_sampling_category["pfas_detected"].eq("Y")
    ]
    .groupby(
        "detection_sampling_event_category",
        dropna=False
    )["pfas_analyte_standardized"]
    .nunique()
    .reset_index(
        name="parameters_positive"
    )
)


# I combine the unique PFAS parameter counts with
# the sampling category summary.

sampling_category_summary = (
    sampling_category_summary
    .merge(
        sampling_parameter_summary,
        on="detection_sampling_event_category",
        how="left"
    )
    .merge(
        sampling_positive_parameter_summary,
        on="detection_sampling_event_category",
        how="left"
    )
)


# I assign zero positive parameters to categories
# where no PFAS analytes were detected.

sampling_category_summary["parameters_positive"] = (
    sampling_category_summary["parameters_positive"]
    .fillna(0)
    .astype(int)
)


display(sampling_category_summary)

,detection_sampling_event_category,locations,testing_instances,sampling_events,sampling_events_with_detection,max_pfas,percent_locations,parameters_tested,parameters_positive
0,No PFAS detected in sampling events,273,8327,363,0,NaN,17.0,140,0
1,PFAS detected in 100% of sampling events,1280,78487,2064,2064,1040000.0,79.9,169,114
2,PFAS detected in less than 100% of sampling events,49,8426,260,189,102.0,3.1,108,46


In [45]:
# I save the sampling-level PFAS detection summary as a CSV file.

sampling_category_summary.to_csv(
    file_path + "saved files/Notebook3/sampling_category_summary.csv",
    index=False
)

print("Sampling category summary saved.")

Sampling category summary saved.


In [46]:
display(
    site_level.loc[
        site_level["detection_sampling_event_category"] == "Unknown",
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "state",
            "testing_instances",
            "parameters_positive",
            "max_pfas"
        ]
    ].head(20)
)

,MonitoringLocationIdentifier,MonitoringLocationName,state,testing_instances,parameters_positive,max_pfas


In [47]:
# I identify the national top 10 PFAS locations.
# I rank locations by the highest PFAS concentration observed
# at each location.

national_top10_locations = (
    site_level
    .loc[
        site_level["max_pfas"].notna()
    ]
    .sort_values(
        "max_pfas",
        ascending=False
    )
    .head(20)
    .copy()
)

display(
    national_top10_locations[
        [
            "MonitoringLocationName",
            "MonitoringLocationIdentifier",
            "state",
            "max_pfas",
            "location_pfas_detected",
            "detection_sampling_event_category",
            "testing_instances",
            "sampling_events",
            "sampling_events_with_detection",
            "percent_sampling_events_with_detection",
            "parameters_tested",
            "parameters_positive",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure"
        ]
    ]
)

,MonitoringLocationName,MonitoringLocationIdentifier,state,max_pfas,location_pfas_detected,detection_sampling_event_category,testing_instances,sampling_events,sampling_events_with_detection,percent_sampling_events_with_detection,parameters_tested,parameters_positive,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure
632,POND IN,MEDEP-157890,ME,1040000.0,Y,PFAS detected in 100% of sampling events,28,1,1,100.0,28,12,43.891285,-69.928219
475,SW-11,MEDEP-144128,ME,197000.0,Y,PFAS detected in 100% of sampling events,745,19,19,100.0,40,19,43.891309,-69.928203
476,SW-17,MEDEP-144131,ME,159000.0,Y,PFAS detected in 100% of sampling events,748,19,19,100.0,40,17,43.889701,-69.922023
477,SW-23,MEDEP-144132,ME,60200.0,Y,PFAS detected in 100% of sampling events,744,19,19,100.0,40,15,43.886078,-69.920747
630,SW-65,MEDEP-157886,ME,43600.0,Y,PFAS detected in 100% of sampling events,743,19,19,100.0,40,18,43.881314,-69.922863
655,MEB,MEDEP-158062,ME,39300.0,Y,PFAS detected in 100% of sampling events,80,2,2,100.0,40,11,43.885182,-69.921077
113,Lake Holloman Deep,21NMEX_WQX-48LHollomanDp,NM,30000.0,Y,PFAS detected in 100% of sampling events,172,2,2,100.0,29,14,32.808220,-106.123980
806,UPPER DETENTION POND,MEDEP-40499,ME,28600.0,Y,PFAS detected in 100% of sampling events,154,6,6,100.0,28,17,44.610290,-69.416121
631,SW-66,MEDEP-157887,ME,14100.0,Y,PFAS detected in 100% of sampling events,746,19,19,100.0,40,14,43.868907,-69.930931
653,SW-67,MEDEP-158058,ME,8160.0,Y,PFAS detected in 100% of sampling events,218,6,6,100.0,40,12,43.861412,-69.936098


In [48]:
national_top10_locations.to_csv(
    file_path +
    "saved files/Notebook3/national_top10_locations.csv",
    index=False
)

print("Saved: national_top10_locations.csv")

Saved: national_top10_locations.csv


In [49]:
# I identify the top 10 PFAS locations in each state.
# Locations are ranked by the highest PFAS concentration observed
# at that location.
# This table will be useful for state case studies and state maps.

top10_locations_per_state = (
    site_level
    .loc[
        site_level["max_pfas"].notna()
    ]
    .sort_values(
        ["state", "max_pfas"],
        ascending=[True, False]
    )
    .groupby(
        "state",
        group_keys=False
    )
    .head(20)
    .copy()
)

display(
    top10_locations_per_state[
        [
            "state",
            "MonitoringLocationName",
            "MonitoringLocationIdentifier",
            "max_pfas",
            "location_pfas_detected",
            "detection_sampling_event_category",
            "testing_instances",
            "sampling_events",
            "sampling_events_with_detection",
            "percent_sampling_events_with_detection",
            "parameters_tested",
            "parameters_positive",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure"
        ]
    ]
)

,state,MonitoringLocationName,MonitoringLocationIdentifier,max_pfas,location_pfas_detected,detection_sampling_event_category,testing_instances,sampling_events,sampling_events_with_detection,percent_sampling_events_with_detection,parameters_tested,parameters_positive,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure
1405,AK,Ship Creek,WKA_2022_0044,7.100000e+00,Y,PFAS detected in 100% of sampling events,54,1,1,100.000000,54,5,61.223720,-149.882170
1464,AL,Neely Henry Lake on the Coosa River,WKA_2022_0115,4.870000e+01,Y,PFAS detected in 100% of sampling events,54,1,1,100.000000,54,13,34.021280,-85.984880
1465,AL,Neely Henry Lake on the Coosa River,WKA_2022_0116,3.760000e+01,Y,PFAS detected in 100% of sampling events,53,1,1,100.000000,53,12,34.020180,-85.988370
1593,AL,Buck Creek,WKA_2022_0265,1.680000e+01,Y,PFAS detected in 100% of sampling events,54,1,1,100.000000,54,9,33.297520,-86.844110
267,AL,Cane Creek Downstream,CAHCaneCreek 08-20-2024-LADown,1.100000e+01,Y,PFAS detected in 100% of sampling events,40,1,1,100.000000,40,9,33.792691,-86.845976
265,AL,Cahaba River Downstream,CAHCahaba 08-22-2024-WDown,1.000000e+01,Y,PFAS detected in 100% of sampling events,40,1,1,100.000000,40,9,33.369614,-86.788120
1484,AL,Mobile River,WKA_2022_0135,8.400000e+00,Y,PFAS detected in 100% of sampling events,54,1,1,100.000000,54,7,30.984930,-87.996460
1485,AL,Mobile River,WKA_2022_0136,8.100000e+00,Y,PFAS detected in 100% of sampling events,54,1,1,100.000000,54,8,30.966990,-87.991210
266,AL,Cahaba River Upstream,CAHCahaba 08-22-2024-WUp,7.300000e+00,Y,PFAS detected in 100% of sampling events,40,1,1,100.000000,40,10,33.369300,-86.785329
268,AL,Cane Creek Downstream,CAHCaneCreek 08-20-2024-LADown-DUP,6.400000e+00,Y,PFAS detected in 100% of sampling events,40,1,1,100.000000,40,9,33.792691,-86.845976


In [50]:
# I summarize the number of monitoring locations by state and detection status.

state_summary = (
    site_level
    .groupby("state", dropna=False)
    .agg(
        total_locations=(
            "MonitoringLocationIdentifier",
            "nunique"
        ),
        detected_locations=(
            "location_pfas_detected",
            lambda x: x.eq("Y").sum()
        ),
        not_detected_locations=(
            "location_pfas_detected",
            lambda x: x.eq("N").sum()
        ),
        uncertain_locations=(
            "location_pfas_detected",
            lambda x: x.eq("U").sum()
        )
    )
    .reset_index()
    .sort_values("state")
)

display(state_summary)
state_summary.to_csv(
    file_path + "saved files/Notebook3/state_summary.csv",
    index=False
)

,state,total_locations,detected_locations,not_detected_locations,uncertain_locations
0,AK,2,1,1,0
1,AL,26,25,1,0
2,AR,2,0,2,0
3,AZ,12,4,8,0
4,CA,101,21,80,0
5,CO,15,9,6,0
6,CT,4,4,0,0
7,D.C.,2,2,0,0
8,DE,15,15,0,0
9,FL,23,22,1,0


In [51]:
# I summarize the number of testing records and monitoring locations by state.

state_record_summary = (
    data_surface
    .groupby("state", dropna=False)
    .agg(
        total_records=(
            "pfas_detected",
            "size"
        ),
        total_locations=(
            "MonitoringLocationIdentifier",
            "nunique"
        ),
        detected_records=(
            "pfas_detected",
            lambda x: x.eq("Y").sum()
        ),
        not_detected_records=(
            "pfas_detected",
            lambda x: x.eq("N").sum()
        ),
        uncertain_records=(
            "pfas_detected",
            lambda x: x.eq("U").sum()
        )
    )
    .reset_index()
    .sort_values("state")
)

display(state_record_summary)

# I save the summary table.

state_record_summary.to_csv(
    file_path + "saved files/Notebook3/state_record_summary.csv",
    index=False
)

,state,total_records,total_locations,detected_records,not_detected_records,uncertain_records
0,AK,108,2,5,103,0
1,AL,1263,26,173,1090,0
2,AR,108,2,0,108,0
3,AZ,836,12,4,832,0
4,CA,1778,101,180,1598,0
5,CO,1442,15,140,1302,0
6,CT,188,4,31,157,0
7,D.C.,108,2,16,92,0
8,DE,720,15,152,568,0
9,FL,1109,23,388,721,0


In [52]:
top10_locations_per_state.to_csv(
    file_path +
    "saved files/Notebook3/pfas_top10_locations_per_state.csv",
    index=False
)

print("Saved: pfas_top10_locations_per_state.csv")
print("Rows:", len(top10_locations_per_state))

Saved: pfas_top10_locations_per_state.csv
Rows: 444


In [53]:
# I review PFAS compounds that were reported as not detected
# at Lake Ontario monitoring locations.

lake_ontario = data_surface[
    data_surface["MonitoringLocationName"]
    .astype("string")
    .str.contains(
        "ONTARIO",
        case=False,
        na=False
    )
].copy()

lake_ontario_non_detects = (
    lake_ontario.loc[
        lake_ontario["pfas_detected"].eq("N")
    ]
    .groupby("pfas_analyte_standardized")
    .size()
    .reset_index(name="non_detect_records")
    .sort_values(
        "non_detect_records",
        ascending=False
    )
)

display(lake_ontario_non_detects.head(30))

,pfas_analyte_standardized,non_detect_records
0,"1-OCTANESULFONIC ACID, 3,3,4,4,5,5,6,6,7,7,8,8,8-TRIDECAFLUORO-",98
1,11-CHLOROEICOSAFLUORO-3-OXAUNDECANE-1-SULFONATE,98
2,2-(N-METHYLPERFLUORO-1-OCTANESULFONAMIDO)-ETHANOL,98
3,3-PERFLUOROHEPTYL PROPANOATE,98
4,3-PERFLUOROPENTYL PROPANOATE,98
5,3-PERFLUOROPROPYL PROPANOATE,98
6,"4,8-DIOXA-3H-PERFLUORONONANOATE",98
7,4:2 FTS,98
8,8:2 FTS,98
9,9CL-PF3ONS,98


In [54]:
# I calculate the number of unique Lake Ontario monitoring locations
# where each PFAS compound was reported as not detected.

lake_ontario_non_detect_locations = (
    lake_ontario.loc[
        lake_ontario["pfas_detected"].eq("N")
    ]
    .groupby("pfas_analyte_standardized")[
        "MonitoringLocationIdentifier"
    ]
    .nunique()
    .reset_index(name="non_detect_locations")
    .sort_values(
        "non_detect_locations",
        ascending=False
    )
)

display(
    lake_ontario_non_detect_locations.head(30)
)

,pfas_analyte_standardized,non_detect_locations
0,"1-OCTANESULFONIC ACID, 3,3,4,4,5,5,6,6,7,7,8,8,8-TRIDECAFLUORO-",26
1,11-CHLOROEICOSAFLUORO-3-OXAUNDECANE-1-SULFONATE,26
2,2-(N-METHYLPERFLUORO-1-OCTANESULFONAMIDO)-ETHANOL,26
3,3-PERFLUOROHEPTYL PROPANOATE,26
4,3-PERFLUOROPENTYL PROPANOATE,26
5,3-PERFLUOROPROPYL PROPANOATE,26
6,"4,8-DIOXA-3H-PERFLUORONONANOATE",26
7,4:2 FTS,26
8,8:2 FTS,26
9,9CL-PF3ONS,26


In [55]:
# I calculate how many unique PFAS compounds were tested
# and how many were detected at least once in the final analysis.

total_pfas_tested = (
    data_surface["pfas_analyte_standardized"]
    .dropna()
    .nunique()
)

total_pfas_detected = (
    data_surface.loc[
        data_surface["pfas_detected"].eq("Y"),
        "pfas_analyte_standardized"
    ]
    .dropna()
    .nunique()
)

print("Unique PFAS compounds tested:", total_pfas_tested)
print("Unique PFAS compounds detected:", total_pfas_detected)

Unique PFAS compounds tested: 174
Unique PFAS compounds detected: 114


In [56]:
# I identify monitoring locations that have exactly the same
# latitude, longitude, and max PFAS compound.

duplicate_location_compound = (
    site_level.loc[
        site_level.duplicated(
            subset=[
                "ActivityLocation/LatitudeMeasure",
                "ActivityLocation/LongitudeMeasure",
                "max_pfas_compound"
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure",
            "max_pfas_compound"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Number of monitoring-location rows with identical "
    "coordinates and max PFAS compound:",
    len(duplicate_location_compound)
)

print(
    "Number of unique duplicated coordinate + compound combinations:",
    duplicate_location_compound[
        [
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure",
            "max_pfas_compound"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

display(
    duplicate_location_compound[
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "state",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure",
            "max_pfas_compound",
            "max_pfas"
        ]
    ]
)

Number of monitoring-location rows with identical coordinates and max PFAS compound: 194
Number of unique duplicated coordinate + compound combinations: 33


,MonitoringLocationIdentifier,MonitoringLocationName,state,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure,max_pfas_compound,max_pfas
0,WKA_2022_0122,Apalachicola River,FL,30.700460,-84.857550,PFOA,6.000
1,WKA_2022_0123,Apalachicola River,FL,30.700460,-84.857550,PFOA,6.500
2,CAPSouthRiver 8-27-24 LADown,South River Downstream,NC,34.990100,-78.643664,PFOS,7.500
3,CAPSouthRiver 8-27-24 LADown DUP,South River upstream,NC,34.990100,-78.643664,PFOS,6.700
4,NARP Pawtuxet - 8/16/24 - WDown,Pawtuxet River Downstream,RI,41.753627,-71.440544,PFOA,7.700
5,NARP Pawtuxet - 8/16/24 - WDown DUP,Pawtuxet River Upstream 1,RI,41.753627,-71.440544,PFOA,7.500
6,20-0220,Cape Cod Bay,MA,41.855180,-70.524500,<NA>,NaN
7,20-0223,Cape Cod Bay,MA,41.855180,-70.524500,<NA>,NaN
8,20-0021,Falls Pond,MA,41.963082,-71.323456,PERFLUOROOCTANESULFONIC ACID,7.700
9,20-0115,Falls Pond,MA,41.963082,-71.323456,PERFLUOROOCTANESULFONIC ACID,9.100


In [57]:
# Based on John's feedback
# I create a state-by-state summary of PFAS monitoring.
# For each state, I calculate:
# - the number of unique monitoring locations tested
# - the number and percentage of locations where PFAS was detected
# - the number of unique PFAS compounds tested
# - the number of unique PFAS compounds detected

In [58]:
# I summarize monitoring locations by state.
# site_level already contains one row per monitoring location.

state_location_summary = (
    site_level
    .groupby("state", dropna=False)
    .agg(
        locations_tested=(
            "MonitoringLocationIdentifier",
            "nunique"
        ),
        locations_with_pfas=(
            "location_pfas_detected",
            lambda x: x.eq("Y").sum()
        )
    )
    .reset_index()
)

In [59]:
# I calculate the percentage of monitoring locations
# where at least one PFAS compound was detected.

state_location_summary[
    "percent_locations_with_pfas"
] = (
    state_location_summary["locations_with_pfas"]
    / state_location_summary["locations_tested"]
    * 100
).round(1)

In [60]:
# I count the number of unique PFAS compounds tested
# in each state.

state_compounds_tested = (
    data_surface
    .groupby(
        "state",
        dropna=False
    )["pfas_analyte_standardized"]
    .nunique()
    .reset_index(
        name="pfas_compounds_tested"
    )
)

In [61]:
# I count the number of unique PFAS compounds detected
# in each state.

state_compounds_found = (
    data_surface.loc[
        data_surface["pfas_detected"].eq("Y")
    ]
    .groupby(
        "state",
        dropna=False
    )["pfas_analyte_standardized"]
    .nunique()
    .reset_index(
        name="pfas_compounds_found"
    )
)

In [62]:
# I combine the location and PFAS compound summaries.

state_by_state_table = (
    state_location_summary
    .merge(
        state_compounds_tested,
        on="state",
        how="left"
    )
    .merge(
        state_compounds_found,
        on="state",
        how="left"
    )
)

In [63]:
# States with no detected PFAS compounds receive a value of zero.

state_by_state_table[
    "pfas_compounds_found"
] = (
    state_by_state_table[
        "pfas_compounds_found"
    ]
    .fillna(0)
    .astype(int)
)

In [64]:
# I sort the final table by state.

state_by_state_table = (
    state_by_state_table
    .sort_values("state")
    .reset_index(drop=True)
)


display(state_by_state_table)



state_by_state_table.to_csv(
    file_path
    + "saved files/Notebook3/state_by_state_pfas_table.csv",
    index=False
)

,state,locations_tested,locations_with_pfas,percent_locations_with_pfas,pfas_compounds_tested,pfas_compounds_found
0,AK,2,1,50.0,54,5
1,AL,26,25,96.2,65,13
2,AR,2,0,0.0,54,0
3,AZ,12,4,33.3,38,1
4,CA,101,21,20.8,79,17
5,CO,15,9,60.0,86,18
6,CT,4,4,100.0,65,10
7,D.C.,2,2,100.0,54,9
8,DE,15,15,100.0,40,15
9,FL,23,22,95.7,78,43


### case studies

In [65]:
# I highlighted three cases studies and I want to use their info
# I create a helper function to summarize PFAS monitoring

def summarize_case_study(case_name, location_mask):

    case_data = data_surface.loc[
        location_mask
    ].copy()

    # I identify the unique monitoring locations included
    # in the case study.

    case_locations = (
        case_data["MonitoringLocationIdentifier"]
        .dropna()
        .unique()
    )

    total_locations = len(case_locations)

    # I identify locations where at least one PFAS compound
    # was detected.

    detected_locations = (
        case_data.loc[
            case_data["pfas_detected"].eq("Y"),
            "MonitoringLocationIdentifier"
        ]
        .dropna()
        .nunique()
    )

    percent_locations_detected = (
        detected_locations / total_locations * 100
        if total_locations > 0
        else pd.NA
    )

    # I count the number of unique PFAS compounds detected
    # anywhere within the case-study area.

    compounds_detected = (
        case_data.loc[
            case_data["pfas_detected"].eq("Y"),
            "pfas_analyte_standardized"
        ]
        .dropna()
        .nunique()
    )

    # I identify the highest quantified PFAS concentration.
    # Only numeric concentration values are used for this metric.

    quantified_detected = case_data.loc[
        case_data["pfas_detected"].eq("Y")
        & case_data["ResultMeasureValue_num"].notna()
    ].copy()

    if len(quantified_detected) > 0:

        max_idx = (
            quantified_detected[
                "ResultMeasureValue_num"
            ]
            .idxmax()
        )

        highest_concentration = (
            quantified_detected.loc[
                max_idx,
                "ResultMeasureValue_num"
            ]
        )

        highest_compound = (
            quantified_detected.loc[
                max_idx,
                "pfas_analyte_standardized"
            ]
        )

        highest_location = (
            quantified_detected.loc[
                max_idx,
                "MonitoringLocationName"
            ]
        )

    else:
        highest_concentration = pd.NA
        highest_compound = pd.NA
        highest_location = pd.NA

    return {
        "case_study": case_name,
        "monitoring_locations": total_locations,
        "locations_with_pfas_detection": detected_locations,
        "percent_locations_with_detection": round(
            percent_locations_detected,
            1
        ),
        "pfas_compounds_detected": compounds_detected,
        "highest_pfas_ng_L": highest_concentration,
        "highest_pfas_compound": highest_compound,
        "highest_pfas_location": highest_location
    }

In [66]:
# Mississippi river

In [67]:
# I inspect monitoring-location names containing Mississippi
# before defining the final Mississippi River case-study area.

mississippi_location_check = (
    data_surface.loc[
        data_surface["MonitoringLocationName"]
        .astype("string")
        .str.contains(
            "Mississippi",
            case=False,
            na=False
        ),
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "state"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "state",
            "MonitoringLocationName"
        ]
    )
)

display(mississippi_location_check)

,MonitoringLocationIdentifier,MonitoringLocationName,state
38561,MNPCA-S016-662,"MISSISSIPPI R 1.75 MI SOUTH OF ST PAUL PARK, MN (RIVER MILE 827)",MN
44156,MNPCA-S009-001,"MISSISSIPPI R 3.8 MI NORTH WEST OF NININGER, MN (RIVER MILE 823)",MN
38548,MNPCA-S003-018,"MISSISSIPPI R AT BOOM ISLAND, AT PLYMOUTH AVE N, MINNEAPOLIS, MN",MN
38564,MNPCA-S001-897,"MISSISSIPPI R AT CSAH 11, 1 MI SW OF BEMIDJI, MN",MN
38699,MNPCA-S000-155,"MISSISSIPPI R AT CSAH-8 7 MI E OF BEMIDJI, MN",MN
43986,MNPCA-S000-151,"MISSISSIPPI R AT MN-115, CAMP RIPLEY, MN",MN
43919,MNPCA-S003-663,"MISSISSIPPI R AT MN-232, 0.3 MI SE OF PALISADE, MN",MN
38535,MNPCA-S006-163,"MISSISSIPPI R DWNSTR OF MN-15 BRG IN SAUK RAPIDS, MN",MN
38851,MNPCA-S006-754,"MISSISSIPPI R JUST OFF SHEPARD RD AT RIVER MILE 842 IN ST. PAUL, MN",MN
44085,MNPCA-S008-062,"MISSISSIPPI R NEAR MOUTH OF TRENTON SLOUGH, 3 MI N. OF RED WING, MN",MN


In [68]:
# I identify PFAS records associated with the Mississippi River.

mississippi_river_mask = (
    data_surface["MonitoringLocationName"]
    .astype("string")
    .str.contains(
        "MISSISSIPPI",
        case=False,
        na=False
    )
)

In [69]:
## Lake Ontario

In [70]:
# I inspect monitoring-location names containing Lake Ontario
# before defining the final Lake Ontario case-study area.

lake_ontario_location_check = (
    data_surface.loc[
        data_surface["MonitoringLocationName"]
        .astype("string")
        .str.contains(
            "Ontario",
            case=False,
            na=False
        ),
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "state"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "state",
            "MonitoringLocationName"
        ]
    )
)

display(lake_ontario_location_check)

,MonitoringLocationIdentifier,MonitoringLocationName,state
23346,USGS-04250773,LAKE ONTARIO (BLACK R BAY) AT SACKETS HARBOR NY,NY
28029,USGS-440152076125301,LAKE ONTARIO (CHAUMONT BAY) AT LONG POINT STATE PK,NY
25328,USGS-435102076122201,LAKE ONTARIO (HENDERSON BAY) AT HENDERSON HARBOR,NY
27840,USGS-432038076424101,LAKE ONTARIO (LITTLE SODUS B) W BARRIER BAR CO PK,NY
24298,USGS-431741077405001,"LAKE ONTARIO (LONG POND) NEAR NORTH GREECE, NY",NY
23765,USGS-431645076494101,"LAKE ONTARIO (PORT BAY) NEAR WOLCOTT, NY",NY
23461,USGS-433756076112001,LAKE ONTARIO (SANDY POND) AT SANDY ISL BEACH ST PK,NY
26090,USGS-431548076593501,"LAKE ONTARIO (SODUS BAY) AT SODUS POINT, NY",NY
23270,USGS-431833077420401,LAKE ONTARIO AT BRADDOCK BAY STATE PARK NY,NY
24945,USGS-431641078591501,"LAKE ONTARIO AT FOURMILE CREEK STATE PARK, NY",NY


In [71]:
# I identify PFAS records associated with Lake Ontario.
# I exclude the Salmon River location at Port Ontario because
# it is a river site rather than a Lake Ontario monitoring location.

lake_ontario_mask = (
    data_surface["MonitoringLocationName"]
    .astype("string")
    .str.contains(
        "ONTARIO",
        case=False,
        na=False
    )
    &
    ~data_surface["MonitoringLocationIdentifier"]
    .eq("USGS-0425050505")
)

In [72]:
lake_ontario_summary = summarize_case_study(
    "Lake Ontario",
    lake_ontario_mask
)

display(
    pd.DataFrame(
        [lake_ontario_summary]
    )
)

,case_study,monitoring_locations,locations_with_pfas_detection,percent_locations_with_detection,pfas_compounds_detected,highest_pfas_ng_L,highest_pfas_compound,highest_pfas_location
0,Lake Ontario,25,25,100.0,8,11.0,PFBA,LK ONTARIO (BRADDOCK BAY) AT BRADDOCK HEIGHTS NY


In [73]:
lake_ontario_locations = (
    data_surface.loc[
        lake_ontario_mask,
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "state"
        ]
    ]
    .drop_duplicates()
    .sort_values("MonitoringLocationName")
)

print(
    "Lake Ontario monitoring locations:",
    lake_ontario_locations[
        "MonitoringLocationIdentifier"
    ].nunique()
)

Lake Ontario monitoring locations: 25


In [74]:
# I check how PFDA and PFUnA were reported across the
# 25 monitoring locations included in the Lake Ontario case study.

lake_ontario_pfas_check = (
    data_surface.loc[
        lake_ontario_mask
        & data_surface["pfas_analyte_standardized"].isin(
            ["PFDA", "PFUnA"]
        ),
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "pfas_analyte_standardized",
            "pfas_detected"
        ]
    ]
    .drop_duplicates()
)

summary = (
    lake_ontario_pfas_check
    .groupby(
        [
            "pfas_analyte_standardized",
            "pfas_detected"
        ],
        dropna=False
    )["MonitoringLocationIdentifier"]
    .nunique()
    .reset_index(
        name="unique_monitoring_locations"
    )
)

display(summary)

,pfas_analyte_standardized,pfas_detected,unique_monitoring_locations
0,PFDA,N,25


In [75]:
## Mississippi river

In [76]:
# I calculate the case-study statistics for the Mississippi River.

mississippi_summary = summarize_case_study(
    "Mississippi River",
    mississippi_river_mask
)

display(
    pd.DataFrame(
        [mississippi_summary]
    )
)

,case_study,monitoring_locations,locations_with_pfas_detection,percent_locations_with_detection,pfas_compounds_detected,highest_pfas_ng_L,highest_pfas_compound,highest_pfas_location
0,Mississippi River,39,39,100.0,13,23.3,PFBA,"MISSISSIPPI R, POOL 2, 1/2 MI NO OF LOCK AND DAM NO. 2, RM 816, HASTINGS, MN"


In [77]:
## Tiger lake

In [78]:
# I identify PFAS records associated with Tiger Lake.

tiger_lake_mask = (
    data_surface["MonitoringLocationName"]
    .astype("string")
    .str.contains(
        "Tiger Lake",
        case=False,
        na=False
    )
)

# I create the Tiger Lake subset.

tiger_lake_data = data_surface.loc[
    tiger_lake_mask
].copy()

# I count the unique monitoring locations included in the case study.

tiger_lake_total_locations = (
    tiger_lake_data[
        "MonitoringLocationIdentifier"
    ]
    .dropna()
    .nunique()
)

# I count monitoring locations where at least one PFAS
# compound was detected.

tiger_lake_detected_locations = (
    tiger_lake_data.loc[
        tiger_lake_data["pfas_detected"].eq("Y"),
        "MonitoringLocationIdentifier"
    ]
    .dropna()
    .nunique()
)

# I calculate the percentage of monitored locations
# with at least one PFAS detection.

tiger_lake_percent_detected = (
    tiger_lake_detected_locations
    / tiger_lake_total_locations
    * 100
)

# I count the unique PFAS compounds detected anywhere
# across the Tiger Lake monitoring locations.

tiger_lake_compounds_detected = (
    tiger_lake_data.loc[
        tiger_lake_data["pfas_detected"].eq("Y"),
        "pfas_analyte_standardized"
    ]
    .dropna()
    .nunique()
)

# I identify the highest quantified PFAS concentration
# among valid detected results.

tiger_lake_quantified = (
    tiger_lake_data.loc[
        tiger_lake_data["pfas_detected"].eq("Y")
        & tiger_lake_data[
            "ResultMeasureValue_num"
        ].notna()
    ]
    .copy()
)

tiger_lake_max_idx = (
    tiger_lake_quantified[
        "ResultMeasureValue_num"
    ]
    .idxmax()
)

tiger_lake_max_value = (
    tiger_lake_quantified.loc[
        tiger_lake_max_idx,
        "ResultMeasureValue_num"
    ]
)

tiger_lake_max_compound = (
    tiger_lake_quantified.loc[
        tiger_lake_max_idx,
        "pfas_analyte_standardized"
    ]
)

tiger_lake_max_location = (
    tiger_lake_quantified.loc[
        tiger_lake_max_idx,
        "MonitoringLocationName"
    ]
)

print(
    "Monitoring locations:",
    tiger_lake_total_locations
)

print(
    "Locations with PFAS detection:",
    tiger_lake_detected_locations
)

print(
    "Percent with PFAS detection:",
    round(tiger_lake_percent_detected, 1)
)

print(
    "Unique PFAS compounds detected:",
    tiger_lake_compounds_detected
)

print(
    "Highest PFAS concentration:",
    tiger_lake_max_value,
    "ng/L"
)

print(
    "Compound with highest concentration:",
    tiger_lake_max_compound
)

print(
    "Location of highest concentration:",
    tiger_lake_max_location
)

Monitoring locations: 6
Locations with PFAS detection: 6
Percent with PFAS detection: 100.0
Unique PFAS compounds detected: 36
Highest PFAS concentration: 19.0 ng/L
Compound with highest concentration: PFBS
Location of highest concentration: Tiger Lake at Marina


In [79]:
# I check for records with exactly the same latitude, longitude,
# and PFAS parameter.

lat_col = "ActivityLocation/LatitudeMeasure"
lon_col = "ActivityLocation/LongitudeMeasure"
parameter_col = "CharacteristicName"

same_coordinates_parameters = (
    data_surface
    .groupby(
        [lat_col, lon_col, parameter_col],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
)

# I keep only combinations that occur more than once.

same_coordinates_parameters = (
    same_coordinates_parameters[
        same_coordinates_parameters["record_count"] > 1
    ]
    .sort_values(
        "record_count",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Number of latitude + longitude + parameter combinations "
    "that appear more than once:",
    len(same_coordinates_parameters)
)

print(
    "Total records belonging to these repeated combinations:",
    same_coordinates_parameters["record_count"].sum()
)

display(same_coordinates_parameters)

Number of latitude + longitude + parameter combinations that appear more than once: 12039
Total records belonging to these repeated combinations: 58044


,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure,CharacteristicName,record_count
0,46.939342,-67.903173,N-METHYL PERFLUOROOCTANE SULFONAMIDOACETIC ACID,135
1,46.939342,-67.903173,N-ETHYL PERFLUOROOCTANE SULFONAMIDOACETIC ACID,134
2,46.939342,-67.903173,PERFLUORONONANOIC ACID,134
3,46.939342,-67.903173,PERFLUOROTRIDECANOIC ACID,134
4,46.939342,-67.903173,PERFLUOROPENTANE SULFONIC ACID,134
5,46.939342,-67.903173,PERFLUOROBUTANOIC ACID,134
6,46.939342,-67.903173,PERFLUORODECANOIC ACID,134
7,46.939342,-67.903173,6:2 FLUOROTELOMER SULFONATE,134
8,46.939342,-67.903173,PERFLUORODODECANOIC ACID,134
9,46.939342,-67.903173,8:2 FLUOROTELOMER SULFONATE,134
